In [7]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from scipy.sparse import hstack, csr_matrix

In [2]:
data = pd.read_csv('./data/preprocessed/enron_spam_data_preprocessed.csv')

In [ ]:
# Hadnle NaN caused by read CSV
data['Subject'] = data['Subject'].fillna("")
data['Message'] = data['Message'].fillna("")

In [ ]:
# Label encoding : Map Spam/Ham to 1/0
y = data['Label'].map({'ham': 0, 'spam': 1})

In [ ]:
# Create Title's number of words feature and Message's number of words feature
meta_features = pd.DataFrame()
meta_features['subject_word_count'] = data['Subject'].apply(lambda x: len(str(x).split()))
meta_features['message_word_count'] = data['Message'].apply(lambda x: len(str(x).split()))
meta_features['url_count'] = data['Message'].apply(lambda x: str(x).count('__url__'))

In [ ]:
# Split data
X_text = data[['Subject', 'Message']]
X_meta = meta_features

X_text_train, X_text_test, X_meta_train, X_meta_test, y_train, y_test = train_test_split(
    X_text, X_meta, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# TF-IDF for Subject feature
tfidf_sub = TfidfVectorizer(min_df=2, max_features=1000)
X_train_sub_tfidf = tfidf_sub.fit_transform(X_text_train['Subject'])
X_test_sub_tfidf = tfidf_sub.transform(X_text_test['Subject'])

In [ ]:
# TF-IDF for Message feature
tfidf_msg = TfidfVectorizer(min_df=5, max_df=0.7, max_features=4000)
X_train_msg_tfidf = tfidf_msg.fit_transform(X_text_train['Message'])
X_test_msg_tfidf = tfidf_msg.transform(X_text_test['Message'])

In [ ]:
# Scaling with Min-Max method
scaler = MinMaxScaler()
X_train_meta_scaled = scaler.fit_transform(X_meta_train)
X_test_meta_scaled = scaler.transform(X_meta_test)

In [12]:
# Feature Selection
X_train_combined = hstack([X_train_sub_tfidf, X_train_msg_tfidf, csr_matrix(X_train_meta_scaled)])
X_test_combined = hstack([X_test_sub_tfidf, X_test_msg_tfidf, csr_matrix(X_test_meta_scaled)])

In [13]:
selector = SelectKBest(chi2, k=3500)
X_train_final = selector.fit_transform(X_train_combined, y_train)
X_test_final = selector.transform(X_test_combined)

In [14]:
X_train_final.shape

(24383, 3500)

In [15]:
X_test_final.shape

(6096, 3500)

In [16]:
os.makedirs('./data/ready_for_train', exist_ok=True)

In [18]:
joblib.dump(X_train_final, './data/ready_for_train/X_train_final.pkl')
joblib.dump(X_test_final, './data/ready_for_train/X_test_final.pkl')
joblib.dump(y_train, './data/ready_for_train/y_train.pkl')
joblib.dump(y_test, './data/ready_for_train/y_test.pkl')

['./data/ready_for_train/y_test.pkl']

In [19]:
joblib.dump(tfidf_sub, './data/ready_for_train/tfidf_subject.pkl')
joblib.dump(tfidf_msg, './data/ready_for_train/tfidf_message.pkl')
joblib.dump(scaler, './data/ready_for_train/scaler.pkl')
joblib.dump(selector, './data/ready_for_train/selector.pkl')

['./data/ready_for_train/selector.pkl']